In [ ]:
# MAX_NEW_TOKENS = 300   
# MAX_INPUT_LEN  = 1024  
# DO_SAMPLE      = False
# TEMPERATURE    = 1.0

In [ ]:
import drafter_pipeline as dp
import transformers_ as tf
import vllm_ 

In [ ]:
from __future__ import annotations

import logging
import re
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import torch
import torch.nn.functional as F
from torch.profiler import ProfilerActivity, profile, record_function
from transformers import AutoModelForCausalLM, AutoTokenizer
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
'''
Specialist RAG drafter : generates m draft (answer, rationale) pairs from m document subsets
in a SINGLE batched forward pass
'''
class BatchedDrafter:
    def __init__(
        self, model_name = None, device= None,
        max_new_tokens = dp.MAX_NEW_TOKENS,
        max_input_len = dp.MAX_INPUT_LEN,
        use_vllm = dp.VLLM_AVAILABLE,
        use_bnb_nf4 = dp.BNB_AVAILABLE
    ):
        if device is not None:
            self.device = torch.device(device)
        elif torch.cuda.is_available():
            self.device = torch.device('cuda')
        elif torch.backends.mps.is_availabe():
            self.device = torch.device('mps')
        else:
            logger.warning('No GPU/MPS - falling back to CPU')
            self.device = torch.device('cpu')
     

        self.is_cuda = self.device.type == 'cuda'
        self.is_mps = self.device.type == 'mps'
        self.is_cpu = self.device.type == 'cpu'

        if model_name is None:
            model_name = dp.MODEL_MISTRAL_7B 

        self.max_input_len = max_input_len
        self.max_new_tokens = max_new_tokens

        self.use_vllm = use_vllm and self.is_cuda
        self.use_bnb_nf4 = use_bnb_nf4 and self.is_cuda

        if use_vllm and not self.use_vllm:
            logger.warning("use_vllm=True ignored: vLLM unavailable or device is not CUDA.")
        if use_bnb_nf4 and not self.use_bnb_nf4:
            logger.warning("use_bnb_nf4=True ignored: bitsandbytes unavailable or device is not CUDA.")

        logger.info(
            'BatchedDrafter : device = %s | model = %s | vllm = %s | bnb_nf4 = %s',
            self.device, model_name, self.use_vllm, self.use_bnb_nf4
        )
        if self.use_vllm:
            vllm_.load_vllm(model_name)
        else:
            tf.load_transformers(model_name)
            
        